Create a offset sine wave script using the identity cosAcosB-sinAsinB = cos(A+B). Note here, this does an offset with the center frequency of transmitter. In order for values stay the same, sample rates must match when actually transmitting. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# --- Settings ---
fs = 10_000_000            # 10 MS/s
f_tone = 500_000           # 0.5 MHz tone
duration = 1.0             # 1 second is plenty
out_path = "./waveforms/sine_tone500KHz_at_fs10MHz_1sec.dat"

# Make sure END matches START in phase:
#   duration * f_tone must be an integer
N_cycles = int(duration * f_tone)
duration = N_cycles / f_tone   # adjust duration to be exact

# Time vector
t = np.arange(int(fs * duration)) / fs

# Generate CLEAN complex sinusoid
amp = 0.7                      # avoid clipping, 70% fullscale
iq = amp * np.exp(1j * 2 * np.pi * f_tone * t)

# Convert to int8 interleaved
I = np.int8(np.real(iq) * 127)
Q = np.int8(np.imag(iq) * 127)

iq_bytes = np.empty(2 * len(I), dtype=np.int8)
iq_bytes[0::2] = I
iq_bytes[1::2] = Q

# Save file
iq_bytes.tofile(out_path)
print("Saved clean, continuous tone to:", out_path)

# N = 2000   # number of samples to plot (adjust if needed)

# plt.figure(figsize=(12,4))
# plt.plot(I[:N], label="I (real part)")
# #plt.plot(Q[:N], label="Q (imag part)")
# plt.title("0.5 MHz Clean Complex Tone (First {} Samples)".format(N))
# plt.xlabel("Sample Index")
# plt.ylabel("Amplitude (int8)")
# plt.legend()
# plt.grid(True)
# plt.show()

Saved clean, continuous tone to: ./waveforms/sine_tone500KHz_at_fs10MHz_1sec.dat


Can also create all IQ all 1's

In [2]:
import numpy as np

# --- Settings ---
fs = 5_000_000              # sample rate (Hz)
symbol_rate = 100_000      # symbols per second -> fs/symrate = 50 sps
symbols_high = 100            # I = 32167, Q = 0 - 1ms for 100
symbols_low  = 900            # I = 0, Q = 0
duration = 3               # total output duration (seconds)
out_path = "./waveforms/hi_low_I_5Mhzfs_3sec.dat"

A = 80 # amplitude (keep <127 to avoid clipping after rotation)
phi_deg = 30
phi = np.deg2rad(phi_deg) # set desired constant phase offset here (e.g., 30 degrees)

# --- Samples per symbol ---
sps = int(fs / symbol_rate)

# --- Build one pattern block of symbols ---
num_symbols = symbols_high + symbols_low

I_symbols = np.zeros(num_symbols, dtype=np.int8)
Q_symbols = np.zeros(num_symbols, dtype=np.int8)

I_symbols[:symbols_high] = A   # first 100 symbols high
# Q stays zero

# --- Expand symbols into samples ---
I_block = np.repeat(I_symbols, sps)
Q_block = np.repeat(Q_symbols, sps)

block_len = len(I_block) # block being chuncks [samples high, samples low]

# --- Determine total samples needed ---
total_I_samples = int(fs * duration)

# --- Repeat block until duration is filled ---
repeats = int(np.ceil(total_I_samples / block_len))

I_full = np.tile(I_block, repeats)[:total_I_samples] 
Q_full = np.tile(Q_block, repeats)[:total_I_samples]

# Apply constant phase offset: (I + jQ) * e^(j phi)
x = I_full + 1j * Q_full
x_rot = x * np.exp(1j * phi)

# Quantize back to int8 safely
I_full_q = np.clip(np.round(np.real(x_rot)), -128, 127).astype(np.int8)
Q_full_q = np.clip(np.round(np.imag(x_rot)), -128, 127).astype(np.int8)


# --- Interleave I/Q int8 ---
iq = np.empty(2*total_I_samples, dtype=np.int8)
iq[0::2] = I_full
iq[1::2] = Q_full

# --- Save ---
iq.tofile(out_path)

print("Saved:", out_path)
print("Duration (s):", duration)
print("Total I samples:", total_I_samples)
print("Symbols per block:", num_symbols)
print("Samples per symbol:", sps)
print("Phase offset (degrees):", phi_deg)


Saved: ./waveforms/hi_low_I_5Mhzfs_3sec.dat
Duration (s): 3
Total I samples: 15000000
Symbols per block: 1000
Samples per symbol: 50
Phase offset (degrees): 30


In [3]:
total_I_samples/fs

3.0